In [3]:
import os
import random
from itertools import combinations

import torch

from dataset import _load_raw_skeleton, preprocess_skeleton
from model import SiameseGaitVerifier


# Hardcoded test configuration
TEST_PATH = "../data/split_casia_b/test/"
CHECKPOINT_PATH = "best_gait_verifier.pth"

SEQUENCE_LENGTH = 64
THRESHOLD = 0.5
EMBEDDING_DIM = 128
HIDDEN_DIM = 256

# How many repeated random rounds to run
NUM_ROUNDS = 3


def load_model(device):
    if not os.path.exists(CHECKPOINT_PATH):
        raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

    model = SiameseGaitVerifier(
        num_nodes=17,
        in_channels=8,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
    ).to(device)

    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)

    model.eval()
    return model


def collect_subject_files():
    if not os.path.exists(TEST_PATH):
        raise FileNotFoundError(f"TEST_PATH not found: {TEST_PATH}")

    subject_to_files = {}
    invalid_files = []

    for subject in sorted(os.listdir(TEST_PATH)):
        subject_path = os.path.join(TEST_PATH, subject)
        if not os.path.isdir(subject_path):
            continue

        valid_files = []
        for root, _, files in os.walk(subject_path):
            for name in files:
                if not name.endswith(".pkl"):
                    continue
                p = os.path.join(root, name)
                try:
                    _load_raw_skeleton(p)
                    valid_files.append(p)
                except Exception:
                    invalid_files.append(p)

        if valid_files:
            subject_to_files[subject] = sorted(valid_files)

    return subject_to_files, invalid_files


def file_to_input(pkl_path, device):
    seq = _load_raw_skeleton(pkl_path)
    x = preprocess_skeleton(seq, sequence_length=SEQUENCE_LENGTH, is_training=False)
    return x.unsqueeze(0).to(device)


def predict_same(model, file1, file2, device):
    x1 = file_to_input(file1, device)
    x2 = file_to_input(file2, device)

    with torch.no_grad():
        logits, z1, z2 = model(x1, x2)
        prob_same = torch.sigmoid(logits)[0, 0].item()
        cosine = torch.nn.functional.cosine_similarity(z1, z2)[0].item()

    pred_same = prob_same >= THRESHOLD
    return pred_same, prob_same, cosine


def evaluate_cross_class(model, subject_to_files, device):
    chosen = {}
    for subject, files in subject_to_files.items():
        chosen[subject] = random.choice(files)

    pairs = list(combinations(sorted(chosen.keys()), 2))
    total = len(pairs)
    correct = 0

    print("\nCross-class test (different people)")
    print("-" * 72)

    for s1, s2 in pairs:
        pred_same, prob, cos = predict_same(model, chosen[s1], chosen[s2], device)
        is_correct = not pred_same
        if is_correct:
            correct += 1

        print(
            f"{s1} vs {s2} | pred={'SAME' if pred_same else 'DIFF'} | "
            f"target=DIFF | prob_same={prob:.4f} | cos={cos:.4f}"
        )

    acc = correct / max(1, total)
    return correct, total, acc


def evaluate_within_class(model, subject_to_files, device):
    eligible = [s for s, files in subject_to_files.items() if len(files) >= 2]
    total = len(eligible)
    correct = 0
    skipped = len(subject_to_files) - len(eligible)

    print("\nWithin-class test (same person)")
    print("-" * 72)

    for subject in sorted(eligible):
        f1, f2 = random.sample(subject_to_files[subject], 2)
        pred_same, prob, cos = predict_same(model, f1, f2, device)
        is_correct = pred_same
        if is_correct:
            correct += 1

        print(
            f"{subject} | pred={'SAME' if pred_same else 'DIFF'} | "
            f"target=SAME | prob_same={prob:.4f} | cos={cos:.4f}"
        )

    acc = correct / max(1, total)
    return correct, total, acc, skipped


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Testing on: {device}")
    print(f"Threshold: {THRESHOLD}")

    subject_to_files, invalid_files = collect_subject_files()
    if len(subject_to_files) == 0:
        raise RuntimeError("No valid subjects found in test folder.")

    print(f"Subjects found: {len(subject_to_files)}")
    print(f"Invalid files skipped: {len(invalid_files)}")

    model = load_model(device)

    cross_correct_all, cross_total_all = 0, 0
    same_correct_all, same_total_all = 0, 0

    for round_idx in range(1, NUM_ROUNDS + 1):
        print("\n" + "=" * 30)
        print(f"Round {round_idx}/{NUM_ROUNDS}")
        print("=" * 30)

        cc, ct, cacc = evaluate_cross_class(model, subject_to_files, device)
        sc, st, sacc, skipped = evaluate_within_class(model, subject_to_files, device)

        cross_correct_all += cc
        cross_total_all += ct
        same_correct_all += sc
        same_total_all += st

        print("\nRound summary")
        print(f"Cross-class accuracy : {cacc:.4f} ({cc}/{ct})")
        print(f"Within-class accuracy: {sacc:.4f} ({sc}/{st})")
        if skipped > 0:
            print(f"Skipped classes for within-class test (need >=2 videos): {skipped}")

    print("\n" + "=" * 30)
    print("Final summary")
    print("=" * 30)
    print(
        f"Cross-class overall : {cross_correct_all / max(1, cross_total_all):.4f} "
        f"({cross_correct_all}/{cross_total_all})"
    )
    print(
        f"Within-class overall: {same_correct_all / max(1, same_total_all):.4f} "
        f"({same_correct_all}/{same_total_all})"
    )


if __name__ == "__main__":
    main()

Testing on: cuda
Threshold: 0.5
Subjects found: 12
Invalid files skipped: 0

Round 1/3

Cross-class test (different people)
------------------------------------------------------------------------
113 vs 114 | pred=DIFF | target=DIFF | prob_same=0.0090 | cos=0.4503
113 vs 115 | pred=DIFF | target=DIFF | prob_same=0.4246 | cos=0.7818
113 vs 116 | pred=DIFF | target=DIFF | prob_same=0.1357 | cos=0.7032
113 vs 117 | pred=DIFF | target=DIFF | prob_same=0.0502 | cos=0.3716
113 vs 118 | pred=DIFF | target=DIFF | prob_same=0.2665 | cos=0.6957
113 vs 119 | pred=SAME | target=DIFF | prob_same=0.6945 | cos=0.8674
113 vs 120 | pred=DIFF | target=DIFF | prob_same=0.2034 | cos=0.6956
113 vs 121 | pred=DIFF | target=DIFF | prob_same=0.0749 | cos=0.6765
113 vs 122 | pred=DIFF | target=DIFF | prob_same=0.0063 | cos=0.5610
113 vs 123 | pred=DIFF | target=DIFF | prob_same=0.1104 | cos=0.5373
113 vs 124 | pred=DIFF | target=DIFF | prob_same=0.1100 | cos=0.6077
114 vs 115 | pred=DIFF | target=DIFF | prob_